# Kalman R6.1 — Decision-Aware Target Ablation v2

Uses frozen R5.0.1 scored rows + frozen R5C0 trade ledger as the authoritative baseline. LIVE R5.1 is untouched.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, pathlib, shutil, subprocess, sys, json

ROOT = pathlib.Path('/content/Kalman_R6_1')
if ROOT.exists():
    shutil.rmtree(ROOT)

subprocess.check_call([
    'git','clone','--depth','1',
    '--branch','research/r6-selective-horizon-20260922',
    'https://github.com/kimtk94/Codex.git', str(ROOT)
])

subprocess.check_call([sys.executable,'-m','pip','install','-q','pyarrow','scikit-learn','joblib','pytest'])

HEAD = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'], text=True).strip()
print('HEAD =', HEAD)

script = ROOT/'kalman-toss-gateway/research/r6_1_decision_target.py'
test = ROOT/'kalman-toss-gateway/tests/test_r6_1_decision_target.py'
subprocess.check_call([sys.executable,'-m','py_compile',str(script)])
subprocess.check_call([sys.executable,'-m','pytest','-q',str(test)])

env = os.environ.copy()
env['KALMAN_DATA_ROOT'] = '/content/drive/MyDrive'
env.pop('R6_ALLOW_LIVE', None)

run = subprocess.run(
    [sys.executable, str(script)],
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(run.stdout)

if run.returncode != 0:
    log = pathlib.Path('/content/drive/MyDrive/US_ETF/model_lab_v1/results/r6_1_decision_target/r6_1_execution_log.json')
    if log.exists():
        print('\n=== LAST R6.1 CHECKPOINT ===')
        print(log.read_text())
    raise RuntimeError(f'R6.1 failed with exit code {run.returncode}')


In [ ]:
import json, pandas as pd

OUT = '/content/drive/MyDrive/US_ETF/model_lab_v1/results/r6_1_decision_target'
leader = pd.read_csv(f'{OUT}/r6_1_leaderboard.csv')
folds = pd.read_csv(f'{OUT}/r6_1_fold_summary.csv')
boot = pd.read_csv(f'{OUT}/r6_1_paired_bootstrap.csv')
sched = pd.read_csv(f'{OUT}/r6_1_schedule_audit.csv')
decision = json.load(open(f'{OUT}/r6_1_selection_decision.json'))
feature_audit = json.load(open(f'{OUT}/r6_1_feature_reconciliation.json'))
path_audit = json.load(open(f'{OUT}/r6_1_path_audit.json'))
training_contract = json.load(open(f'{OUT}/r6_1_training_contract.json'))

print('=== BASELINE RECONCILIATION ===')
print(json.dumps(decision['baseline_reconciliation'], indent=2))

print('\n=== TRAINING CONTRACT ===')
print(json.dumps(training_contract, indent=2))

print('\n=== FEATURE RECONCILIATION ===')
print(json.dumps({k:v for k,v in feature_audit.items() if k != 'details'}, indent=2))

print('\n=== PATH AUDIT ===')
print(json.dumps(path_audit, indent=2))

print('\n=== SCHEDULE AUDIT ===')
display(sched)

print('\n=== R6.1 LEADERBOARD ===')
display(leader)

print('\n=== BOOTSTRAP ===')
display(boot)

print('\n=== FOLD SUMMARY ===')
display(folds)

print('\n=== DECISION ===')
print('survivors =', decision['research_survivors'])
print('selected  =', decision['selected_r6_1_candidate'])
print('promotion_eligible =', decision['promotion_eligible'])
